# 08 머신러닝 예측 모델 (Phase 1 — 2/3)

**40조건** × **RF, XGBoost** | 지표: MAE, RMSE, MAPE, MASE

### ⓪ 환경 설정

In [ ]:
import sys
from pathlib import Path
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'code' else NOTEBOOK_DIR
sys.path.insert(0, str(REPO_ROOT / 'code'))

from utils.paths import DATA_PROCESSED
from utils.splits import VAL_WEEKS, TRAIN_WEEK_MAX
from utils.device import device_label
from utils.experiment_data import load_forecast_frames
from utils.phase_experiments import (
    STAT_MODELS, ML_MODELS, DL_MODELS, RANK_METRIC,
    run_phase1_all, summarize_phase1, merge_phase1_best,
    build_global_embedding_cache, run_phase2_all, summarize_phase2,
    pick_final_per_condition,
)

print('Torch device:', device_label())
df, feat_df = load_forecast_frames()
print('시계열:', df.groupby(['type', 'family']).ngroups)
print('학습 <=', TRAIN_WEEK_MAX, '| 검증:', VAL_WEEKS)
print('Best 선정 기준:', RANK_METRIC.upper())


### ① ML 모델 실험

In [ ]:
# Phase 1 — ML 2종: SBC 20조건 + ML 20조건
cache = DATA_PROCESSED / 'phase1_ml_results.parquet'
if cache.exists():
    results = pd.read_parquet(cache)
    summary, best = summarize_phase1(results)
    print('캐시 로드 |', len(results), 'rows')
else:
    sbc = run_phase1_all(df, feat_df, 'SBC_CLUSTER', 'SBC', models=ML_MODELS)
    ml = run_phase1_all(df, feat_df, 'ML_CLUSTER', 'ML', models=ML_MODELS)
    results = pd.concat([sbc, ml], ignore_index=True)
    summary, best = summarize_phase1(results)
    results.to_parquet(cache, index=False)
    summary.to_csv(DATA_PROCESSED / 'phase1_ml_results_summary.csv', index=False)
    best.to_csv(DATA_PROCESSED / 'phase1_ml_results_best.csv', index=False)
    print('완료 |', len(results), 'rows')

display(best.sort_values(['cluster_scheme', 'type', 'cluster']))
print('\n=== 알고리즘별 평균', RANK_METRIC.upper(), '===')
print(results.groupby(['cluster_scheme', 'model'])[RANK_METRIC].mean().unstack('cluster_scheme').round(2))
